In [3]:
import pandas as pd
import numpy as np
import os

# Set seed for reproducibility
np.random.seed(42)
num_customers = 5000

# 1. Generate Demographics Data
customer_ids = [f"CUST-{i:05d}" for i in range(1, num_customers + 1)]

demographics = pd.DataFrame({
    'CustomerID': customer_ids,
    'SignupDate': pd.date_range(start='2023-01-01', periods=num_customers, freq='h')[:num_customers],
    'ContractType': np.random.choice(['Month-to-month', 'One year', 'Two year'], num_customers, p=[0.6, 0.2, 0.2]),
    'PaymentMethod': np.random.choice(['Electronic Check', 'Mailed Check', 'Bank Transfer', 'Credit Card'], num_customers),
    'Age': np.random.randint(18, 80, size=num_customers)
})

# Inject missing values into 'PaymentMethod' to simulate real-world data gaps
demographics.loc[demographics.sample(frac=0.05).index, 'PaymentMethod'] = np.nan

# 2. Generate Activity Data
activity = pd.DataFrame({
    'CustomerID': customer_ids,
    'MonthlyCharges': np.random.uniform(29.99, 120.00, size=num_customers).round(2),
    'TotalCharges': np.zeros(num_customers), # Will compute this logically later
    'SupportTickets': np.random.poisson(lam=1.5, size=num_customers),
    'EngagementScore': np.random.randint(1, 11, size=num_customers),
    'LeftCompany': np.random.choice([0, 1], num_customers, p=[0.77, 0.23])
})

# Add extreme billing outliers (e.g., accidental $5,000 charges due to system bugs)
outlier_indices = activity.sample(n=15).index
activity.loc[outlier_indices, 'MonthlyCharges'] = 5000.00

# Save these as raw files
os.makedirs('data/raw', exist_ok=True)
demographics.to_csv('data/raw/customer_demographics.csv', index=False)
activity.to_csv('data/raw/customer_activity.csv', index=False)

print("Raw datasets successfully generated inside 'data/raw/'!")

Raw datasets successfully generated inside 'data/raw/'!


In [4]:
# Step 2: ETL Pipeline (Extract, Transform, Load)
import pandas as pd
import numpy as np
import os
from datetime import datetime

print("Starting ETL Process...")

# 1. EXTRACT: Load the raw data we just created
demographics = pd.read_csv('data/raw/customer_demographics.csv')
activity = pd.read_csv('data/raw/customer_activity.csv')

# Convert SignupDate from a text string back into a time-aware format
demographics['SignupDate'] = pd.to_datetime(demographics['SignupDate'])

# 2. TRANSFORM (MERGE): Combine the tables
df = pd.merge(demographics, activity, on='CustomerID', how='inner')

# 3. TRANSFORM (CLEAN): Handle Missing Values
df['PaymentMethod'] = df['PaymentMethod'].fillna('Unknown')

# 4. TRANSFORM (FILTER): Remove Outliers
df = df[df['MonthlyCharges'] < 1000]

# 5. FEATURE ENGINEERING: Create new, useful columns
# We'll use today's date (July 5, 2026) as our reference point to calculate tenure
current_date = pd.to_datetime('2026-07-05')

# Calculate 'TenureMonths': Total days divided by 30
df['TenureMonths'] = ((current_date - df['SignupDate']).dt.days / 30).round()

# Calculate 'TotalCharges': Monthly Bill * Months stayed
df['TotalCharges'] = df['MonthlyCharges'] * df['TenureMonths']

# 6. LOAD: Save the cleaned and processed data
os.makedirs('data/processed', exist_ok=True)
df.to_csv('data/processed/clean_customer_data.csv', index=False)

print(f"ETL Complete! Final dataset has {df.shape[0]} rows and {df.shape[1]} columns.")
print("Clean data saved to 'data/processed/clean_customer_data.csv'")

Starting ETL Process...
ETL Complete! Final dataset has 4985 rows and 11 columns.
Clean data saved to 'data/processed/clean_customer_data.csv'


In [1]:
# Step 3: Machine Learning & Predictive Modeling
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
import os

print("Starting Machine Learning Phase...")

# 1. Load the clean data we just created in Step 2
df = pd.read_csv('data/processed/clean_customer_data.csv')

# 2. Prepare the data for the AI (Encoding)
# AI models are basically math equations; they don't understand text like "Credit Card".
# We use a LabelEncoder to translate text categories into numerical codes (e.g., 0, 1, 2).
label_encoder = LabelEncoder()
df['ContractType_Num'] = label_encoder.fit_transform(df['ContractType'])
df['PaymentMethod_Num'] = label_encoder.fit_transform(df['PaymentMethod'])

# 3. Select Features (X) and Target (y)
# X = The "biomarkers" or clues we give the AI to learn from
features = ['Age', 'MonthlyCharges', 'TotalCharges', 'SupportTickets', 
            'EngagementScore', 'TenureMonths', 'ContractType_Num', 'PaymentMethod_Num']
X = df[features]

# y = What we want the AI to diagnose/predict (1 = Left Company, 0 = Stayed)
y = df['LeftCompany']

# 4. Split data into "Study Material" (Train) and "Exam" (Test)
# We hide 20% of the data from the AI so we can test its accuracy later.
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 5. Build and Train the Model
# We are using a Random Forest with 100 "trees"
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# 6. Grade the AI's "Exam"
accuracy = model.score(X_test, y_test)
print(f"AI Model Successfully Trained! Accuracy on hidden test data: {accuracy * 100:.2f}%")

# 7. Generate "Churn Risk Scores" for ALL customers
# Instead of just saying "Yes" or "No", we ask the AI for a probability percentage (0% to 100%)
churn_probabilities = model.predict_proba(X)[:, 1] 
df['ChurnRiskScore_%'] = (churn_probabilities * 100).round(2)

# 8. Save the final "Gold" dataset for Power BI Dashboarding
os.makedirs('data/final', exist_ok=True)
df.to_csv('data/final/powerbi_customer_data.csv', index=False)

print("Machine Learning Complete! Final Power BI dataset saved to 'data/final/powerbi_customer_data.csv'")

Starting Machine Learning Phase...
AI Model Successfully Trained! Accuracy on hidden test data: 74.32%
Machine Learning Complete! Final Power BI dataset saved to 'data/final/powerbi_customer_data.csv'
